# 06 - FAIR Data Conversion and Export

Bruker files are proprietary binary. `epyr.fair` exports them to open formats
(CSV, JSON, HDF5) with standardized metadata, following FAIR principles
(Findable, Accessible, Interoperable, Reusable).

- `convert_bruker_to_fair(input, output_dir, formats=...)` -> `bool`
- `batch_convert_directory(in_dir, out_dir, ...)`
- `validate_fair_dataset(data_dict, path)` -> `ValidationResult`

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")  # keep tutorial output readable

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import epyr

DATA = Path("..") / "data"   # example datasets, relative to this notebook
print("EPyR Tools version:", epyr.__version__)

In [ ]:
import tempfile, shutil
from epyr.fair import convert_bruker_to_fair, batch_convert_directory, validate_fair_dataset

work = Path(tempfile.mkdtemp(prefix="epyr_fair_"))
print("working directory:", work)

## Convert a single file to CSV, JSON, and HDF5

In [ ]:
out_single = work / "single"
ok = convert_bruker_to_fair(
    DATA / "130406SB_CaWO4_Er_CW_5K_20.DSC",
    output_dir=out_single,
    formats=["csv", "json", "hdf5"])

print("conversion succeeded:", ok)
for f in sorted(out_single.iterdir()):
    print(f"  {f.name:40s} {f.stat().st_size:>8d} bytes")

## Validate a dataset

`validate_fair_dataset` checks data integrity and FAIR compliance. Raw Bruker
metadata lacks several FAIR fields (title, creator, license, ...), so freshly
loaded data is reported as not yet compliant. This is the point of the check: it
lists exactly what `convert_bruker_to_fair` needs to add. `get_summary()` gives a
one-line overview.

In [ ]:
x, y, params, fp = epyr.eprload(DATA / "130406SB_CaWO4_Er_CW_5K_20.DSC",
                                plot_if_possible=False)
report = validate_fair_dataset({"x_data": x, "y_data": y, "metadata": params}, Path(fp))

summary = report.get_summary()
print(f"FAIR-compliant as loaded: {report.is_valid}")
print(f"errors: {summary['error_count']}, warnings: {summary['warning_count']}")
print("\nMissing required fields:")
for err in report.errors[:5]:
    print("  -", err)

## Batch conversion

`batch_convert_directory` converts every Bruker file in a directory. Here a small
temporary input directory is built from one file pair to keep the demo fast.

In [ ]:
in_dir = work / "batch_in"
in_dir.mkdir()
for ext in (".par", ".spc"):
    shutil.copy(DATA / f"CuSO4_001{ext}", in_dir / f"CuSO4_001{ext}")

out_dir = work / "batch_out"
batch_convert_directory(in_dir, out_dir, file_extensions=[".par"],
                        output_formats=["csv", "json"])

for f in sorted(out_dir.rglob("*")):
    if f.is_file():
        print(" ", f.relative_to(out_dir))

In [ ]:
# Clean up the temporary working directory
shutil.rmtree(work, ignore_errors=True)
print("cleaned up", work)

## Summary

- `convert_bruker_to_fair` exports one file; `batch_convert_directory` handles a folder.
- Formats are any subset of `csv`, `json`, `hdf5` (and `jpg` previews).
- `validate_fair_dataset` confirms integrity and FAIR compliance before sharing.